In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# --- Canales y Utilidades ---
def ebno_to_noise_std(
    ebno_db: float, bits_per_symbol: int, num_complex_dims: int = 1
) -> float:
  ebno_linear = 10.0 ** (ebno_db / 10.0)
  code_rate = bits_per_symbol / (2.0 * num_complex_dims)
  noise_variance = 1.0 / (2.0 * code_rate * ebno_linear)
  return np.sqrt(noise_variance)


class AWGNChannel(nn.Module):

  def __init__(self):
    super(AWGNChannel, self).__init__()

  def forward(self, x: torch.Tensor, noise_std: float) -> torch.Tensor:
    noise = torch.randn_like(x) * noise_std
    return x + noise


# --- Módulos Autoencoder ---
class Transmitter(nn.Module):

  def __init__(self, num_symbols: int = 16, channel_dims: int = 2):
    super(Transmitter, self).__init__()
    self.M = num_symbols
    self.fc1 = nn.Linear(self.M, 32)
    self.fc2 = nn.Linear(32, 32)
    self.fc3 = nn.Linear(32, channel_dims)

  def forward(self, s_onehot: torch.Tensor) -> torch.Tensor:
    x = F.relu(self.fc1(s_onehot))
    x = F.relu(self.fc2(x))
    x_raw = self.fc3(x)
    energy = torch.mean(torch.sum(x_raw**2, dim=1, keepdim=True))
    return x_raw / torch.sqrt(energy)


class Receiver(nn.Module):

  def __init__(
      self, num_symbols: int = 16, channel_dims: int = 2, use_csi: bool = False
  ):
    super(Receiver, self).__init__()
    self.use_csi = use_csi
    input_dim = channel_dims * 2 if use_csi else channel_dims
    self.fc1 = nn.Linear(input_dim, 32)
    self.fc2 = nn.Linear(32, 32)
    self.fc3 = nn.Linear(32, num_symbols)

  def forward(self, y: torch.Tensor, h: torch.Tensor = None) -> torch.Tensor:
    inp = torch.cat([y, h], dim=1) if (self.use_csi and h is not None) else y
    x = F.relu(self.fc1(inp))
    x = F.relu(self.fc2(x))
    return self.fc3(x)


class EndToEndAutoencoder(nn.Module):

  def __init__(
      self, num_symbols: int = 16, channel_dims: int = 2, use_csi: bool = False
  ):
    super(EndToEndAutoencoder, self).__init__()
    self.M = num_symbols
    self.transmitter = Transmitter(num_symbols, channel_dims)
    self.receiver = Receiver(num_symbols, channel_dims, use_csi)


# --- Sanity Check & Backpropagation Test ---
M = 16
batch_size = 1024
ebno_train = 7.0

model = EndToEndAutoencoder(num_symbols=M, channel_dims=2, use_csi=False)
channel = AWGNChannel()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

symbols = torch.randint(0, M, (batch_size,))
one_hot = F.one_hot(symbols, num_classes=M).float()

x_norm = model.transmitter(one_hot)
noise_std = ebno_to_noise_std(ebno_db=ebno_train, bits_per_symbol=4)
y_received = channel(x_norm, noise_std)
logits = model.receiver(y_received)

loss = criterion(logits, symbols)

optimizer.zero_grad()
loss.backward()
optimizer.step()

energy_per_symbol = torch.mean(torch.sum(x_norm**2, dim=1)).item()

print(f"Pérdida (Loss) Inicial: {loss.item():.4f}")
print(f"Energía Media Transmitida E[||x||^2]: {energy_per_symbol:.4f}")
print(f"Dimensiones de salida del Transmisor: {x_norm.shape}")
print(f"Dimensiones de salida del Receptor: {logits.shape}")

assert (
    abs(energy_per_symbol - 1.0) < 1e-4
), "¡Error! La normalización de energía falló."
print("\n✅ Sanity Check completado con éxito.")